# Clustering Gene Expression Data: ALL vs. AML Leukemia

This notebook is designed for a senior undergraduate course, **Applied Data Mining for Bioinformatics**.

## Learning goals
By the end of this notebook, students should be able to:

1. load and reshape a real gene expression dataset,
2. explain why clustering is an **unsupervised** learning task,
3. preprocess high-dimensional expression data for clustering,
4. apply **K-means** and **hierarchical clustering**,
5. interpret clusters biologically using known leukemia labels **after** clustering,
6. recognize why unsupervised results depend on preprocessing and distance definitions.

## Dataset
We use the classic leukemia microarray dataset derived from Golub et al. (1999), *Molecular classification of cancer: class discovery and class prediction by gene expression monitoring*.

The files used here are public CSV versions of:
- `data_set_ALL_AML_train.csv`
- `data_set_ALL_AML_independent.csv`
- `data_set_label.csv`

The notebook first looks for local copies of these files in the same folder. If they are not present, it downloads them from a public GitHub repository that republishes the dataset for teaching and demonstration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from urllib.request import urlretrieve

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_context("talk")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 10)

In [ ]:
# File names and fallback URLs
base = Path(".")

files = {
    "train": {
        "name": "data_set_ALL_AML_train.csv",
        "url": "https://raw.githubusercontent.com/ProfNascimento/MP/main/data_set_ALL_AML_train.csv",
    },
    "independent": {
        "name": "data_set_ALL_AML_independent.csv",
        "url": "https://raw.githubusercontent.com/ProfNascimento/MP/main/data_set_ALL_AML_independent.csv",
    },
    "labels": {
        "name": "data_set_label.csv",
        "url": "https://raw.githubusercontent.com/ProfNascimento/MP/main/data_set_label.csv",
    },
}

for item in files.values():
    path = base / item["name"]
    if not path.exists():
        print(f"Downloading {item['name']} ...")
        urlretrieve(item["url"], path)
    else:
        print(f"Found local file: {item['name']}")

In [ ]:
train_df = pd.read_csv(files["train"]["name"])
ind_df = pd.read_csv(files["independent"]["name"])
labels_df = pd.read_csv(files["labels"]["name"])

print("Training table shape:", train_df.shape)
print("Independent table shape:", ind_df.shape)
print("Label table shape:", labels_df.shape)

train_df.head()

## 1. Understanding the raw layout

This dataset is stored in a format that is common in older microarray teaching datasets:

- each **row** is a gene (or probe set),
- the first two columns describe the gene/probe,
- the remaining columns alternate between:
  - an expression value column for a sample, and
  - a `call` column that stores a detection call.

For clustering **samples**, we want a matrix of shape:

- rows = samples
- columns = genes

In [ ]:
# Sample columns are the columns whose names are numeric strings.
train_sample_cols = [c for c in train_df.columns if c.isdigit()]
ind_sample_cols = [c for c in ind_df.columns if c.isdigit()]

print("Number of training samples:", len(train_sample_cols))
print("Number of independent samples:", len(ind_sample_cols))
print("Training sample IDs:", train_sample_cols[:10], "...")
print("Independent sample IDs:", ind_sample_cols[:10], "...")

In [ ]:
# Keep the two gene annotation columns and the numeric sample columns only.
combined_df = pd.concat(
    [
        train_df[["Gene Description", "Gene Accession Number"] + train_sample_cols],
        ind_df[["Gene Description", "Gene Accession Number"] + ind_sample_cols],
    ],
    axis=1,
)

# Remove duplicated metadata columns created by concatenation.
combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]

print("Combined gene x sample table shape:", combined_df.shape)
combined_df.head()

In [ ]:
# Build sample-by-gene matrix
sample_cols = train_sample_cols + ind_sample_cols

X = combined_df[sample_cols].T
X.index = X.index.astype(int)
X = X.sort_index()

# Match labels to sample order
y = labels_df.set_index("patient").loc[X.index, "cancer"]

print("Sample-by-gene matrix shape:", X.shape)
print()
print("Class counts:")
print(y.value_counts())
X.iloc[:5, :5]

## 2. Basic preprocessing

### Why preprocessing matters
Gene expression data are **high dimensional**:
- many thousands of genes,
- relatively few samples.

That means clustering can be strongly affected by:
- control probes,
- very low-variance genes,
- scale differences across genes.

In this notebook we use a simple, teachable pipeline:

1. remove Affymetrix control probes (`AFFX...`),
2. keep the **top 100 most variable genes**,
3. standardize each gene across samples.

This is not the only possible pipeline, but it is a reasonable classroom choice.

In [ ]:
# Remove Affymetrix control probes
non_control_mask = ~combined_df["Gene Accession Number"].astype(str).str.startswith("AFFX")
X_nocontrol = X.loc[:, non_control_mask.values]

print("Shape after removing control probes:", X_nocontrol.shape)

In [ ]:
# Select the top 100 most variable genes across samples
gene_variances = X_nocontrol.var(axis=0)
top_n = 100
top_gene_idx = gene_variances.sort_values(ascending=False).head(top_n).index

X_top = X_nocontrol.loc[:, top_gene_idx]

print("Shape after selecting top variable genes:", X_top.shape)
print("Top 10 variances:")
gene_variances.sort_values(ascending=False).head(10)

In [ ]:
# Standardize genes (columns) so that each gene has mean 0 and variance 1 across samples
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_top)

X_scaled_df = pd.DataFrame(X_scaled, index=X_top.index, columns=X_top.columns)
X_scaled_df.iloc[:5, :5]

## 3. Quick exploratory view with PCA

PCA is **not** a clustering method, but it is very useful for visualization.
We project the 100-dimensional sample representation down to 2 dimensions so we can see broad structure.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    {
        "PC1": X_pca[:, 0],
        "PC2": X_pca[:, 1],
        "Cancer": y.values,
        "Sample": X_top.index,
    }
)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
pca_df.head()

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cancer", s=120)
plt.title("PCA of leukemia gene expression samples")
plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

### Interpretation
If samples of the same leukemia type tend to appear near one another in the PCA plot, that suggests there may be real underlying structure in the data.

But remember:
- PCA is a visualization tool,
- clustering will still be performed separately.

## 4. K-means clustering

K-means tries to partition samples into K groups so that samples are close to the centroid of their assigned cluster.

Here we choose **K = 2** because the known biology contains two leukemia categories:
- ALL
- AML

In a real unsupervised workflow, the true labels would not be used to fit the clustering model. We use them **only afterward** to help interpret the result.

In [ ]:
# Silhouette scores for several K values
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_k = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels_k)

pd.Series(sil_scores, name="Silhouette score")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()), marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("K-means silhouette scores")
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=20)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_results = pd.DataFrame(
    {
        "Sample": X_top.index,
        "Cancer": y.values,
        "KMeansCluster": kmeans_labels,
    }
)

kmeans_results.head(10)

In [ ]:
# Because cluster labels 0 and 1 are arbitrary, use ARI to compare clustering with true labels.
true_binary = (y == "AML").astype(int).values
ari_kmeans = adjusted_rand_score(true_binary, kmeans_labels)

print(f"Adjusted Rand Index for K-means: {ari_kmeans:.3f}")
print()
print("Contingency table:")
pd.crosstab(kmeans_results["Cancer"], kmeans_results["KMeansCluster"])

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=kmeans_labels,
    style=y.values,
    s=120,
)
plt.title("PCA view colored by K-means cluster")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

### Teaching note
K-means often works best when clusters are roughly:
- compact,
- spherical,
- similar in size.

Gene expression data do not always satisfy those assumptions. So even when K-means is useful, it may not perfectly recover biological classes.

## 5. Hierarchical clustering

Hierarchical clustering is especially common in bioinformatics because it produces a **dendrogram**, which is easy to interpret visually.

Here we use:
- Euclidean distance on the standardized top-variable genes
- **complete linkage**

Complete linkage often produces tighter, more compact clusters than single linkage.

In [ ]:
Z = linkage(X_scaled, method="complete", metric="euclidean")

plt.figure(figsize=(16, 6))
dendrogram(
    Z,
    labels=[f"{sample}:{label}" for sample, label in zip(X_top.index, y.values)],
    leaf_rotation=90,
)
plt.title("Hierarchical clustering dendrogram")
plt.xlabel("Sample")
plt.ylabel("Linkage distance")
plt.tight_layout()
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=2, linkage="complete")
agg_labels = agg.fit_predict(X_scaled)

ari_agg = adjusted_rand_score(true_binary, agg_labels)

print(f"Adjusted Rand Index for hierarchical clustering: {ari_agg:.3f}")
print()
print("Contingency table:")
pd.crosstab(y.values, agg_labels, rownames=["Cancer"], colnames=["HierCluster"])

## 6. Heatmap of top variable genes

Heatmaps are a classic way to display clustered gene expression data.

To keep the figure readable in class, we show only the **top 40 most variable genes**.  
Rows are samples, columns are genes.

In [ ]:
top_heatmap_n = 40
heatmap_gene_idx = gene_variances.sort_values(ascending=False).head(top_heatmap_n).index

X_heatmap = X_nocontrol.loc[:, heatmap_gene_idx]
X_heatmap_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_heatmap),
    index=X_heatmap.index,
    columns=X_heatmap.columns,
)

# Order samples by hierarchical cluster labels and then by sample index
sample_order = pd.DataFrame(
    {"sample": X_heatmap_scaled.index, "cluster": agg_labels, "Cancer": y.values}
).sort_values(["cluster", "Cancer", "sample"])["sample"]

X_heatmap_scaled = X_heatmap_scaled.loc[sample_order]

plt.figure(figsize=(16, 8))
sns.heatmap(X_heatmap_scaled, cmap="vlag", center=0)
plt.title("Heatmap of 40 most variable genes (samples ordered by hierarchical clusters)")
plt.xlabel("Genes")
plt.ylabel("Samples")
plt.tight_layout()
plt.show()

## 7. What did we learn?

### Main observations
1. The ALL/AML leukemia samples contain structure that can be explored without using labels during fitting.
2. PCA helps visualize broad separation patterns.
3. Different clustering algorithms can behave differently on the same data.
4. Results depend strongly on preprocessing choices such as:
   - removing controls,
   - selecting variable genes,
   - standardization,
   - distance/linkage choice.

### Important bioinformatics lesson
Clustering does **not** prove biology by itself.  
It suggests patterns that should then be examined with:
- pathway analysis,
- differential expression,
- clinical metadata,
- external validation.

## 8. Suggested student exercises

1. Change the number of selected genes from 100 to 50, 200, and 500.  
   How do the PCA plot and clustering results change?

2. Try other linkage rules:
   - `ward`
   - `average`
   - `single`

3. Try clustering with **correlation distance** instead of Euclidean distance.  
   Why might correlation be attractive for gene expression data?

4. Repeat K-means several times with different random seeds.  
   Does the result change?

5. Replace the top-variable-gene filter with another rule, such as median absolute deviation (MAD).  
   How sensitive are the findings?